In [1]:
# ===== CORE PYTHON =====
import os
import sys
import random
import shutil
import json

# ===== DATA HANDLING =====
import numpy as np
import pandas as pd
from glob import glob
from collections import Counter

# ===== IMAGE / AUDIO =====
import cv2
import librosa
import soundfile as sf
from PIL import Image

# ===== TORCH =====
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

# ===== ML / METRICS =====
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

# ===== PROGRESS =====
from tqdm import tqdm

# ===== YAML CONFIG =====
import yaml

# ===== OPTIONAL (SAFE) =====
import warnings
warnings.filterwarnings("ignore")

# ===== COLAB ENV =====
from google.colab import drive

print("All core libraries loaded successfully.")

All core libraries loaded successfully.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, random, yaml, shutil
from glob import glob
from collections import defaultdict

random.seed(42)
os.environ["WANDB_MODE"] = "disabled"

!git clone -q https://github.com/xaCheng1996/MVF.git /content/MVF || true
%pip install -q dotmap einops timm pyyaml pydub yacs

print("Setup done")

Mounted at /content/drive
Setup done


In [3]:
# UnZip FakeAVCeleb Data
!mkdir -p /content/dataset
!unzip -q -o "/content/drive/MyDrive/FakeAVCeleb_v1.2.zip" -d /content/dataset

print("Videos:", len(glob("/content/dataset/**/*.mp4", recursive=True)))
print(glob("/content/dataset/**/*.mp4", recursive=True)[:3])

Videos: 21544
['/content/dataset/FakeAVCeleb_v1.2/FakeVideo-RealAudio/Caucasian (American)/women/id00231/00149_0.mp4', '/content/dataset/FakeAVCeleb_v1.2/FakeVideo-RealAudio/Caucasian (American)/women/id00231/00037_id00462_wavtolip.mp4', '/content/dataset/FakeAVCeleb_v1.2/FakeVideo-RealAudio/Caucasian (American)/women/id00231/00037_id00418_KSTOz7H0OuQ.mp4']


In [4]:
# PATCH MVF REPO  1) make RandAugment optional
aug_path = "/content/MVF/utils/Augmentation.py"
with open(aug_path, "r") as f:
    lines = f.readlines()

fixed = [
    "from datasets.transforms_ss import *\n",
    "try:\n",
    "    from randaugment import RandAugment\n",
    "except Exception:\n",
    "    RandAugment = None\n",
]

start_copy = 0
for idx, line in enumerate(lines):
    if line.startswith("class ") or line.startswith("def "):
        start_copy = idx
        break

fixed.extend(lines[start_copy:])

with open(aug_path, "w") as f:
    f.writelines(fixed)

# 2) safer split for file paths
ds_path = "/content/MVF/datasets/datasets.py"
with open(ds_path, "r") as f:
    code = f.read()

code = code.replace(".split(' ')", ".rsplit(' ', 1)")

with open(ds_path, "w") as f:
    f.write(code)

print("Repo patched")
!sed -n '1,12p' /content/MVF/utils/Augmentation.py
!grep -n "rsplit(' ', 1)" /content/MVF/datasets/datasets.py | head

Repo patched
from datasets.transforms_ss import *
try:
    from randaugment import RandAugment
except Exception:
    RandAugment = None
class GroupTransform(object):
    def __init__(self, transform):
        self.worker = transform

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]

124:        self.video_list = [VideoRecord(x.strip().rsplit(' ', 1)) for x in open(self.list_file)]
236:        self.video_list = [FrameRecord(x.strip().rsplit(' ', 1)) for x in open(self.list_file)]
304:        self.audio_list = [FrameRecord(x.strip().rsplit(' ', 1)) for x in open(self.list_file)]
374:        self.audio_list = [FrameRecord(x.strip().rsplit(' ', 1)) for x in open(self.list_wav_file)]
375:        self.video_list = [FrameRecord(x.strip().rsplit(' ', 1)) for x in open(self.list_video_file)]
499:        self.video_list = [FrameRecord(x.strip().rsplit(' ', 1)) for x in open(self.list_face_file)]
711:        self.video_list = [FrameRecord(x.strip().rsplit(

In [5]:
# DATA PREPARATION - Generate MVF training/validation files
official_cfg = "/content/MVF/configs/FakeAVCeleb/finetune.yaml"
new_cfg = "/content/MVF/configs/FakeAVCeleb/finetune_full.yaml"

with open(official_cfg, "r") as f:
    cfg = yaml.safe_load(f)

cfg["resume"] = ""
cfg["pretrain"]["split"] = False
cfg["pretrain"]["pretrain"] = ""

cfg["data"]["train_face_list"] = "/content/MVF_data/train_frame.txt"
cfg["data"]["val_face_list"] = "/content/MVF_data/val_frame.txt"
cfg["data"]["label_list"] = "/content/MVF_data/FakeAVCeleb.csv"

cfg["data"]["batch_size"] = 8
cfg["data"]["workers"] = 0
cfg["data"]["neg_sample"] = 1
cfg["data"]["randaug"]["N"] = 0
cfg["data"]["randaug"]["M"] = 0
cfg["data"]["exp_name"] = "FakeAVCeleb_full_train"
cfg["solver"]["lr"] = 1e-4
cfg["solver"]["epochs"] = 10

with open(new_cfg, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

print("Saved:", new_cfg)
print("lr =", cfg["solver"]["lr"])
print("epochs =", cfg["solver"]["epochs"])
!grep -n "train_face_list\|val_face_list\|label_list\|batch_size\|workers\|neg_sample\|epochs" -A 2 "$new_cfg"

Saved: /content/MVF/configs/FakeAVCeleb/finetune_full.yaml
lr = 0.0001
epochs = 10
13:  batch_size: 8
14:  workers: 0
15-  num_classes: 400
16-  image_tmpl: img_{:05d}.jpg
17:  train_face_list: /content/MVF_data/train_frame.txt
18:  val_face_list: /content/MVF_data/val_frame.txt
19:  label_list: /content/MVF_data/FakeAVCeleb.csv
20-  index_bias: 1
21-  input_size: 224
22:  neg_sample: 1
23-  randaug:
24-    N: 0
--
59:  epochs: 10
60-  start_epoch: 0
61-  epoch_offset: 0


In [6]:
# PHASE TRAINING SET UP - Create finetune_full.yaml
import os, random, re, shutil
from glob import glob
from collections import defaultdict

random.seed(42)

def clean_name(s: str) -> str:
    s = s.replace(" ", "_").replace("(", "").replace(")", "")
    s = re.sub(r"[^A-Za-z0-9._-]", "_", s)
    return s

# find FakeAVCeleb root
candidates = [
    "/content/dataset/FakeAVCeleb_v1.2",
    "/content/dataset/FakeAVCeleb_v1.2/FakeAVCeleb_v1.2",
    "/content/dataset",
]

fakeav_base = None
for p in candidates:
    vids = glob(os.path.join(p, "**", "*.mp4"), recursive=True)
    if len(vids) > 1000:
        fakeav_base = p
        break

assert fakeav_base is not None, "FakeAVCeleb root not found"
print("Using fakeav_base:", fakeav_base)

mvf_root = "/content/MVF_data"
face_root = os.path.join(mvf_root, "face")
voice_root = os.path.join(mvf_root, "voice")
train_list = os.path.join(mvf_root, "train_frame.txt")
val_list = os.path.join(mvf_root, "val_frame.txt")
label_csv = os.path.join(mvf_root, "FakeAVCeleb.csv")

if os.path.exists(mvf_root):
    shutil.rmtree(mvf_root)
os.makedirs(face_root, exist_ok=True)
os.makedirs(voice_root, exist_ok=True)

CATEGORY_TO_BUCKET = {
    "RealVideo-RealAudio": ("real", 0),
    "RealVideo-FakeAudio": ("fake_audio", 1),
    "FakeVideo-RealAudio": ("fake_video", 2),
    "FakeVideo-FakeAudio": ("fake_video", 2),
}

groups = defaultdict(lambda: {"real": [], "fake_audio": [], "fake_video": []})

for category, (bucket, label) in CATEGORY_TO_BUCKET.items():
    vids = glob(os.path.join(fakeav_base, category, "**", "*.mp4"), recursive=True)
    for vp in vids:
        rel = os.path.relpath(vp, fakeav_base)
        parts = rel.split(os.sep)

        if len(parts) < 5:
            continue

        ethnicity = clean_name(parts[1])
        gender = clean_name(parts[2])
        identity = clean_name(parts[3])
        identity_key = f"{ethnicity}__{gender}__{identity}"

        groups[identity_key][bucket].append((vp, label, parts))

eligible = {
    k: v for k, v in groups.items()
    if len(v["real"]) > 0 and len(v["fake_audio"]) > 0 and len(v["fake_video"]) > 0
}

print("Eligible identities:", len(eligible))
assert len(eligible) > 0, "No eligible identities found"

keys = list(eligible.keys())
random.shuffle(keys)
split_idx = int(0.8 * len(keys))
train_ids = keys[:split_idx]
val_ids = keys[split_idx:]

MAX_PER_BUCKET = 20

def build_records(id_list, out_records):
    for identity_key in id_list:
        bucket_dict = eligible[identity_key]
        for bucket_name in ["real", "fake_audio", "fake_video"]:
            items = bucket_dict[bucket_name][:MAX_PER_BUCKET]
            for vp, label, parts in items:
                category = clean_name(parts[0])
                ethnicity = clean_name(parts[1])
                gender = clean_name(parts[2])
                identity = clean_name(parts[3])
                filename = clean_name(os.path.splitext(parts[4])[0])

                face_dir = os.path.join(face_root, category, ethnicity, gender, identity)
                voice_dir = os.path.join(voice_root, category, ethnicity, gender, identity)
                os.makedirs(face_dir, exist_ok=True)
                os.makedirs(voice_dir, exist_ok=True)

                img_path = os.path.join(face_dir, filename + ".jpg")
                wav_path = os.path.join(voice_dir, filename + ".wav")

                if not os.path.exists(img_path):
                    os.system(f'ffmpeg -y -i "{vp}" -frames:v 1 "{img_path}" -loglevel error')
                if not os.path.exists(wav_path):
                    os.system(f'ffmpeg -y -i "{vp}" -ar 16000 -ac 1 -vn "{wav_path}" -loglevel error')

                if os.path.exists(img_path) and os.path.exists(wav_path):
                    out_records.append((img_path, label))

train_records, val_records = [], []
build_records(train_ids, train_records)
build_records(val_ids, val_records)

with open(train_list, "w") as f:
    for p, y in train_records:
        f.write(f"{p} {y}\n")

with open(val_list, "w") as f:
    for p, y in val_records:
        f.write(f"{p} {y}\n")

with open(label_csv, "w") as f:
    f.write("id;name\n")
    f.write("0;real\n")
    f.write("1;fake_audio\n")
    f.write("2;fake_video\n")

print("Train identities:", len(train_ids))
print("Val identities:", len(val_ids))
print("Train samples:", len(train_records))
print("Val samples:", len(val_records))
print("Example train:", open(train_list).readline().strip())
print("Example val:", open(val_list).readline().strip())

Using fakeav_base: /content/dataset/FakeAVCeleb_v1.2
Eligible identities: 500
Train identities: 400
Val identities: 100
Train samples: 8781
Val samples: 2200
Example train: /content/MVF_data/face/RealVideo-RealAudio/Asian_South/men/id00350/00015.jpg 0
Example val: /content/MVF_data/face/RealVideo-RealAudio/Caucasian_American/women/id00835/00195.jpg 0


In [7]:
!wc -l /content/MVF_data/train_frame.txt
!wc -l /content/MVF_data/val_frame.txt


8781 /content/MVF_data/train_frame.txt
2200 /content/MVF_data/val_frame.txt


In [8]:
import sys
sys.path.insert(0, "/content/MVF")

from datasets.datasets import FakeAVCeleb_Datasets
from utils.Augmentation import get_augmentation
from dotmap import DotMap
import yaml

cfg_path = "/content/MVF/configs/FakeAVCeleb/finetune_full.yaml"

with open(cfg_path, "r") as f:
    cfg = DotMap(yaml.safe_load(f))

train_data = FakeAVCeleb_Datasets(
    cfg.data.train_face_list,
    cfg.data.label_list,
    neg_sample=cfg.data.neg_sample,
    random_seed=cfg.seed,
    transformer=get_augmentation(True, cfg),
)

print("len(train_data):", len(train_data))

use seed 1024
len(train_data): 400


In [9]:
# training - running finetune_full.yaml.py
import os, sys

os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTHONPATH"] = "/content/MVF:" + os.environ.get("PYTHONPATH", "")

%cd /content/MVF
!{sys.executable} finetune_deepfake.py --config /content/MVF/configs/FakeAVCeleb/finetune_full.yaml --log_time fullrun1

/content/MVF
/opt/bin:/usr/local/cuda/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/tools/node/bin:/tools/google-cloud-sdk/bin
['.', '/opt/bin', '/usr/local/cuda/bin', '/usr/local/sbin', '/usr/local/bin', '/usr/sbin', '/usr/bin', '/sbin', '/bin', '/tools/node/bin', '/tools/google-cloud-sdk/bin', '/opt/anaconda3/envs/torch_cv/bin/']
--------------------------------------------------------------------------------
                     working dir: ./exp/Tranf_face/Transf/FakeAVCeleb/FakeAVCeleb_full_train/fullrun1
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                               Config
{   'data': {   'batch_size': 8,
                'dataset': {   'train_dataset': 'FakeAVCeleb',
                               'val_dataset': 'FakeAVCeleb'},
                'exp_name': 'FakeAVCeleb_full_train',
                'image_tmpl': 'img_{:05d}.jpg',
   

In [10]:
# find checkpoint under /content/MVF/exp/.../last_model.pt
!find /content/MVF/exp -type f | grep -i "last_model"

/content/MVF/exp/Tranf_face/Transf/FakeAVCeleb/FakeAVCeleb_full_train/fullrun1/last_model.pt


In [11]:
# create test_full.yaml
import yaml
import copy

train_cfg_path = "/content/MVF/configs/FakeAVCeleb/finetune_full.yaml"
eval_cfg_path = "/content/MVF/configs/FakeAVCeleb/test_full.yaml"
ckpt_path = "/content/MVF/exp/Tranf_face/Transf/FakeAVCeleb/FakeAVCeleb_full_train/fullrun1/last_model.pt"

with open(train_cfg_path, "r") as f:
    train_cfg = yaml.safe_load(f)   # plain dict

eval_cfg = copy.deepcopy(train_cfg)

eval_cfg["resume"] = ""
eval_cfg["pretrain"]["split"] = False
eval_cfg["pretrain"]["pretrain"] = ckpt_path

eval_cfg["data"]["dataset"] = "FakeAVCeleb"
eval_cfg["data"]["train_face_list"] = "/content/MVF_data/train_frame.txt"
eval_cfg["data"]["val_face_list"] = "/content/MVF_data/val_frame.txt"
eval_cfg["data"]["label_list"] = "/content/MVF_data/FakeAVCeleb.csv"
eval_cfg["data"]["batch_size"] = 4
eval_cfg["data"]["workers"] = 0
eval_cfg["data"]["neg_sample"] = 0
eval_cfg["data"]["randaug"]["N"] = 0
eval_cfg["data"]["randaug"]["M"] = 0
eval_cfg["data"]["exp_name"] = "FakeAVCeleb_full_train"

eval_cfg["solver"]["evaluate"] = True

with open(eval_cfg_path, "w") as f:
    yaml.safe_dump(eval_cfg, f, sort_keys=False)

print("Saved:", eval_cfg_path)
!sed -n '1,80p' "$eval_cfg_path"

Saved: /content/MVF/configs/FakeAVCeleb/test_full.yaml
resume: ''
pretrain:
  split: false
  pretrain: /content/MVF/exp/Tranf_face/Transf/FakeAVCeleb/FakeAVCeleb_full_train/fullrun1/last_model.pt
seed: 1024
data:
  dataset: FakeAVCeleb
  modality: RGB
  num_segments: 1
  seg_length: 1
  batch_size: 4
  workers: 0
  num_classes: 400
  image_tmpl: img_{:05d}.jpg
  train_face_list: /content/MVF_data/train_frame.txt
  val_face_list: /content/MVF_data/val_frame.txt
  label_list: /content/MVF_data/FakeAVCeleb.csv
  index_bias: 1
  input_size: 224
  neg_sample: 0
  randaug:
    N: 0
    M: 0
  random_shift: true
  exp_name: FakeAVCeleb_full_train
network:
  arch: Transf
  init: true
  tsm: false
  drop_out: 0.0
  emb_dropout: 0.0
  MM: true
  layers: 12
  type: Tranf_face
  sim_header: Transf
  joint: false
  describe: null
  id_num: 5994
  voice:
    input_resolution:
    - 128
    - 256
    patch_size: 32
    width: 768
    layers: 12
    heads: 12
    output_dim: 512
  face:
    input_reso

In [12]:
# RUN EVALUATION - test_vfd.py
import os, sys

os.environ["WANDB_MODE"] = "disabled"
os.environ["PYTHONPATH"] = "/content/MVF:" + os.environ.get("PYTHONPATH", "")

%cd /content/MVF
!{sys.executable} test_vfd.py --config /content/MVF/configs/FakeAVCeleb/test_full.yaml --log_time eval_full

/content/MVF
--------------------------------------------------------------------------------
                     working dir: ./exp/Tranf_face/Transf/eval_full
--------------------------------------------------------------------------------
--------------------------------------------------------------------------------
                               Config
{   'data': {   'batch_size': 4,
                'dataset': 'FakeAVCeleb',
                'exp_name': 'FakeAVCeleb_full_train',
                'image_tmpl': 'img_{:05d}.jpg',
                'index_bias': 1,
                'input_size': 224,
                'label_list': '/content/MVF_data/FakeAVCeleb.csv',
                'modality': 'RGB',
                'neg_sample': 0,
                'num_classes': 400,
                'num_segments': 1,
                'randaug': {'M': 0, 'N': 0},
                'random_shift': True,
                'seg_length': 1,
                'train_face_list': '/content/MVF_data/train_frame.txt',

In [13]:
mvf_full_result = {
    "model": "MVF baseline",
    "train_samples": 4791,
    "val_samples": 1200,
    "epochs": 10,
    "batch_size": 8,
    "auc": 0.5680636363636363,
    "accuracy": 0.5308333333333334,
    "checkpoint": "/content/MVF/exp/Tranf_face/Transf/FakeAVCeleb/FakeAVCeleb_full_train/fullrun1/last_model.pt"
}

print(mvf_full_result)

{'model': 'MVF baseline', 'train_samples': 4791, 'val_samples': 1200, 'epochs': 10, 'batch_size': 8, 'auc': 0.5680636363636363, 'accuracy': 0.5308333333333334, 'checkpoint': '/content/MVF/exp/Tranf_face/Transf/FakeAVCeleb/FakeAVCeleb_full_train/fullrun1/last_model.pt'}


In [14]:
# SAVE RESUKT JSON + CHECKPOINT TO DRIVE
import os, json, shutil

save_dir = "/content/drive/MyDrive/TRM_DATASETS/mvf_baseline_full"
os.makedirs(save_dir, exist_ok=True)

ckpt_path = "/content/MVF/exp/Tranf_face/Transf/FakeAVCeleb/FakeAVCeleb_full_train/fullrun1/last_model.pt"

result = {
    "model": "MVF baseline",
    "setup": "fuller training run",
    "train_samples": 4791,
    "val_samples": 1200,
    "epochs": 10,
    "batch_size": 8,
    "auc": 0.5680636363636363,
    "accuracy": 0.5308333333333334,
    "checkpoint": ckpt_path
}

with open("/content/mvf_full_result.json", "w") as f:
    json.dump(result, f, indent=2)

shutil.copy("/content/mvf_full_result.json", os.path.join(save_dir, "mvf_full_result.json"))
shutil.copy(ckpt_path, os.path.join(save_dir, "last_model.pt"))

print("Saved files to:", save_dir)
print(os.listdir(save_dir))

Saved files to: /content/drive/MyDrive/TRM_DATASETS/mvf_baseline_full
['last_model.pt', 'mvf_full_result.json']


# MLP BASELINE - It gives you a simple reference model.

That helps answer:

can a basic model already solve part of the task?
how much better is MVF?
how much better is your final TRM model?
If your TRM beats MLP by a lot, that strengthens your paper.

In [15]:
import os
import numpy as np
from PIL import Image
from torchvision import transforms

train_file = "/content/MVF_data/train_frame.txt"
val_file = "/content/MVF_data/val_frame.txt"

# simple preprocessing:
# resize image to smaller size so MLP training is manageable
img_size = 64

transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),   # converts to [C, H, W] with values in [0,1]
])

def load_face_data(list_file, max_samples=None):
    X, y = [], []

    with open(list_file, "r") as f:
        lines = f.readlines()

    if max_samples is not None:
        lines = lines[:max_samples]

    for line in lines:
        line = line.strip()
        if not line:
            continue

        img_path, label = line.rsplit(" ", 1)

        try:
            img = Image.open(img_path).convert("RGB")
            img_tensor = transform(img)              # [3, 64, 64]
            img_vector = img_tensor.numpy().reshape(-1)   # flatten to 1D vector

            X.append(img_vector)
            y.append(int(label))
        except Exception as e:
            continue

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)
    return X, y

# start with a manageable subset for debugging
X_train, y_train = load_face_data(train_file, max_samples=3000)
X_val, y_val = load_face_data(val_file, max_samples=1000)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)
print("Unique train labels:", np.unique(y_train))
print("Unique val labels:", np.unique(y_val))

X_train shape: (3000, 12288)
y_train shape: (3000,)
X_val shape: (1000, 12288)
y_val shape: (1000,)
Unique train labels: [0 1 2]
Unique val labels: [0 1 2]


In [20]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# device
device = "cuda" if torch.cuda.is_available() else "cpu"

# convert numpy arrays to torch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

# build datasets
train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)

# class weights
class_counts = np.bincount(y_train)
class_weights = len(y_train) / (len(class_counts) * class_counts)
weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

# dataloaders
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

# simple MLP model
class SimpleMLP(nn.Module):
    def __init__(self, input_dim=12288, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)

weighted_mlp_model = SimpleMLP(
    input_dim=X_train.shape[1],
    num_classes=len(np.unique(y_train))
).to(device)

print("Device:", device)
print(weighted_mlp_model)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

Class counts: [ 137  137 2726]
Class weights: [7.29927007 7.29927007 0.36683786]
Device: cuda
SimpleMLP(
  (net): Sequential(
    (0): Linear(in_features=12288, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=128, out_features=3, bias=True)
  )
)
Train batches: 47
Val batches: 16


In [22]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(weighted_mlp_model.parameters(), lr=1e-3)

num_epochs = 10

train_losses = []
val_accuracies = []
val_f1s = []
val_bal_accuracies = []

for epoch in range(num_epochs):
    # ----- training -----
    weighted_mlp_model.train()
    running_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        outputs = weighted_mlp_model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ----- validation -----
    weighted_mlp_model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            outputs = weighted_mlp_model(xb)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_bal_acc = balanced_accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average="macro")

    val_accuracies.append(val_acc)
    val_bal_accuracies.append(val_bal_acc)
    val_f1s.append(val_f1)

    print(f"Epoch [{epoch+1}/{num_epochs}] | Train Loss: {avg_train_loss:.4f} | "
          f"Val Acc: {val_acc:.4f} | Val Bal Acc: {val_bal_acc:.4f} | Val Macro F1: {val_f1:.4f}")

Epoch [1/10] | Train Loss: 1.4417 | Val Acc: 0.0460 | Val Bal Acc: 0.3333 | Val Macro F1: 0.0293
Epoch [2/10] | Train Loss: 1.1706 | Val Acc: 0.0850 | Val Bal Acc: 0.3339 | Val Macro F1: 0.0580
Epoch [3/10] | Train Loss: 1.1196 | Val Acc: 0.9080 | Val Bal Acc: 0.3333 | Val Macro F1: 0.3173
Epoch [4/10] | Train Loss: 1.1144 | Val Acc: 0.9080 | Val Bal Acc: 0.3333 | Val Macro F1: 0.3173
Epoch [5/10] | Train Loss: 1.1145 | Val Acc: 0.3510 | Val Bal Acc: 0.3352 | Val Macro F1: 0.1987
Epoch [6/10] | Train Loss: 1.1140 | Val Acc: 0.9080 | Val Bal Acc: 0.3333 | Val Macro F1: 0.3173
Epoch [7/10] | Train Loss: 1.1038 | Val Acc: 0.2980 | Val Bal Acc: 0.3433 | Val Macro F1: 0.1771
Epoch [8/10] | Train Loss: 1.1040 | Val Acc: 0.9080 | Val Bal Acc: 0.3333 | Val Macro F1: 0.3173
Epoch [9/10] | Train Loss: 1.1082 | Val Acc: 0.8870 | Val Bal Acc: 0.3325 | Val Macro F1: 0.3230
Epoch [10/10] | Train Loss: 1.1102 | Val Acc: 0.9080 | Val Bal Acc: 0.3333 | Val Macro F1: 0.3173


In [23]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

# weighted_mlp_model.load_state_dict(torch.load("/content/best_weighted_mlp.pth"))
weighted_mlp_model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for xb, yb in val_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        outputs = weighted_mlp_model(xb)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

acc = accuracy_score(all_labels, all_preds)
balanced_acc = balanced_accuracy_score(all_labels, all_preds)
macro_f1 = f1_score(all_labels, all_preds, average="macro")
macro_precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
macro_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
cm = confusion_matrix(all_labels, all_preds)

print("=== WEIGHTED MLP (CLASS WEIGHTS ONLY) FINAL EVALUATION ===")
print("Accuracy:", round(acc, 4))
print("Balanced Accuracy:", round(balanced_acc, 4))
print("Macro Precision:", round(macro_precision, 4))
print("Macro Recall:", round(macro_recall, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4, zero_division=0))

=== WEIGHTED MLP (CLASS WEIGHTS ONLY) FINAL EVALUATION ===
Accuracy: 0.908
Balanced Accuracy: 0.3333
Macro Precision: 0.3027
Macro Recall: 0.3333
Macro F1: 0.3173

Confusion Matrix:
[[  0   0  46]
 [  0   0  46]
 [  0   0 908]]

Classification Report:
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000        46
           1     0.0000    0.0000    0.0000        46
           2     0.9080    1.0000    0.9518       908

    accuracy                         0.9080      1000
   macro avg     0.3027    0.3333    0.3173      1000
weighted avg     0.8245    0.9080    0.8642      1000



In [24]:
import pandas as pd

mlp_results = pd.DataFrame([
    {
        "Model": "MLP Baseline",
        "Setting": "Standard CE loss",
        "Accuracy": 0.9080,
        "Balanced Accuracy": 0.3333,
        "Macro Precision": 0.3027,
        "Macro Recall": 0.3333,
        "Macro F1": 0.3173,
        "Behavior": "Collapsed to class 2"
    },
    {
        "Model": "MLP Variant",
        "Setting": "Class weights only",
        "Accuracy": 0.9080,
        "Balanced Accuracy": 0.3333,
        "Macro Precision": 0.3027,
        "Macro Recall": 0.3333,
        "Macro F1": 0.3173,
        "Behavior": "Collapsed to class 2"
    },
    {
        "Model": "MLP Variant",
        "Setting": "Class weights + sampler",
        "Accuracy": 0.0460,
        "Balanced Accuracy": 0.3333,
        "Macro Precision": 0.0153,
        "Macro Recall": 0.3333,
        "Macro F1": 0.0293,
        "Behavior": "Collapsed to class 0"
    }
])

print(mlp_results)

          Model                  Setting  Accuracy  Balanced Accuracy  \
0  MLP Baseline         Standard CE loss     0.908             0.3333   
1   MLP Variant       Class weights only     0.908             0.3333   
2   MLP Variant  Class weights + sampler     0.046             0.3333   

   Macro Precision  Macro Recall  Macro F1              Behavior  
0           0.3027        0.3333    0.3173  Collapsed to class 2  
1           0.3027        0.3333    0.3173  Collapsed to class 2  
2           0.0153        0.3333    0.0293  Collapsed to class 0  


AV HUBERT

In [25]:
%cd /content
!git clone https://github.com/facebookresearch/av_hubert.git
%cd /content/av_hubert
!git submodule update --init --recursive
!pip install -r requirements.txt

/content
Cloning into 'av_hubert'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 149 (delta 61), reused 52 (delta 52), pack-reused 43 (from 1)
Receiving objects: 100% (149/149), 4.64 MiB | 18.34 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/av_hubert
Submodule 'fairseq' (https://github.com/pytorch/fairseq) registered for path 'fairseq'
Cloning into '/content/av_hubert/fairseq'...
Submodule path 'fairseq': checked out 'afc77bdf4bb51453ce76f1572ef2ee6ddcda8eeb'
Submodule 'fairseq/model_parallel/megatron' (https://github.com/ngoyal2707/Megatron-LM) registered for path 'fairseq/fairseq/model_parallel/megatron'
Cloning into '/content/av_hubert/fairseq/fairseq/model_parallel/megatron'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (6/6), done.
remote: Total 8 (delta 6), reused 6 (delta 6), pack-reused 2 (from 1)
Unpacking objects: 100% (8/8), 1.85 

In [26]:
!ls /content/av_hubert
!ls /content/av_hubert/avhubert
!ls /content/av_hubert/avhubert/preparation

assets	  CODE_OF_CONDUCT.md  fairseq  README.md
avhubert  CONTRIBUTING.md     LICENSE  requirements.txt
clustering     hubert_criterion.py    infer_s2s.py  resnet.py
conf	       hubert_dataset.py      __init__.py   sequence_generator.py
decoder.py     hubert_pretraining.py  misc	    utils.py
hubert_asr.py  hubert.py	      preparation
align_mouth.py	       detect_landmark.py  lrs3_prepare.py    requirements.txt
count_frames.py        gen_subword.py	   musan_prepare.py   vox_manifest.py
count_frames_slurm.py  lrs3_manifest.py    noise_manifest.py  vox_prepare.py
data		       lrs3_noise.py	   README.md


In [27]:
!sed -n '1,200p' /content/av_hubert/avhubert/preparation/lrs3_manifest.py

# Copyright (c) Facebook, Inc. and its affiliates.
# All rights reserved.
#
# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.

import os
import glob
import shutil
import subprocess
from tqdm import tqdm
from pathlib import Path
from gen_subword import gen_vocab
from tempfile import NamedTemporaryFile

def main():
    import argparse
    parser = argparse.ArgumentParser(description='LRS3 tsv preparation', formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    parser.add_argument('--lrs3', type=str, help='lrs3 root dir')
    parser.add_argument('--valid-ids', type=str, help='a list of valid ids')
    parser.add_argument('--vocab-size', type=int, default=1000, help='a list of valid ids')
    args = parser.parse_args()
    file_list, label_list = f"{args.lrs3}/file.list", f"{args.lrs3}/label.list"
    assert os.path.isfile(file_list) , f"{file_list} not exist -> run lrs3_prepare.py first"
    assert os.path.isfile(l

In [28]:
import pandas as pd

df = pd.read_csv('/content/MVF_data/FakeAVCeleb.csv')
print(df.columns.tolist())
print(df.head(5))
print(df.shape)

['id;name']
        id;name
0        0;real
1  1;fake_audio
2  2;fake_video
(3, 1)
